# Adaptive hysteresis stepping

This notebook runs the same small single-grain demonstration as `adaptive_hysteresis_demo.py`. The fixed 0.01 T sweep is the reference. Adaptive sweeps should follow the reference near reversal while accepting far fewer field states away from it.

The calculation functions remain in the Python file so the script and notebook cannot develop different numerical conventions.

In [ ]:
from pathlib import Path
import sys

# Support opening Jupyter either in this directory or at the repository root.
candidate_directories = [
    Path.cwd(),
    Path.cwd() / 'python' / 'tests' / 'adaptive_hysteresis',
]
demo_directory = next(
    directory
    for directory in candidate_directories
    if (directory / 'adaptive_hysteresis_demo.py').is_file()
)
if str(demo_directory) not in sys.path:
    sys.path.insert(0, str(demo_directory))

import adaptive_hysteresis_demo as demo

## Parameters

Field values and step sizes below are $\mu_0H$ in tesla. `cells_per_axis=5` gives 125 magnetic cells, allowing spatial variation while keeping the demonstration reasonably quick.

In [ ]:
config = demo.DemoConfig(
    grain_size_m=10.0e-9,
    cells_per_axis=5,
    saturation_induction_t=2.4,
    anisotropy_j_per_m3=1.0e6,
    exchange_j_per_m=7.0e-12,
    field_tilt_deg=3.0,
    field_start_t=1.0,
    field_end_t=-2.0,
    fixed_steps_t=(0.5, 0.1, 0.01),
    adaptive_initial_step_t=0.5,
    adaptive_max_step_t=0.5,
    adaptive_min_steps_t=(0.1, 0.01),
    adaptive_max_accepted_steps=512,
)
config

## Run the five hysteresis curves

Each call creates a fresh magnetic problem, so all curves begin from the same positively saturated state.

In [ ]:
runs = demo.run_all_hysteresis(config)
[(demo.run_label(run), run['n_evaluations']) for run in runs]

## Quantitative comparison

The table reports accepted field counts, coercivity differences, and piecewise-linear magnetisation errors relative to the fixed 0.01 T curve.

In [ ]:
metrics = demo.comparison_metrics(runs, reference_step_t=0.01)
demo.print_metrics_table(metrics)

## Curves, error, and accepted step sizes

The bottom panel is the clearest view of adaptive behaviour: large steps are retained away from switching, while small steps are inserted around reversal.

In [ ]:
figure, axes = demo.plot_comparison(runs, reference_step_t=0.01)
figure